# Packages

In [2]:
# Basic Package Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Non-basic package imports
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import requests

# Packages I don't understand
from fcd_torch import FCD
import rdkit
from collections import Counter
import gc
import pickle
import wandb

# Add the Python_files directory to the Python path
import sys
import os
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'Python_files'))

# Now you can import your modules
import functions_enc as f
import function_depot as fd

# Weightes Loss Function Code Drafts

I'd adding here a few drafts of code that we could use to implement the weighted loss function I have been looking at.

In [ ]:
# Here is the code I got from copilot defining the weighted loss function, as a 2 two step function:
# In this T is the benchmark value, and alpha is the weight applied to values below T.
def weighted_loss(y_pred, y_true, alpha):
    T = torch.log(torch.tensor(100.0))  # ln(100)
    # base loss (MSE here, but could be MAE or Huber)
    base_loss = (y_pred - y_true) ** 2

    # weight function
    weights = torch.where(
        y_true <= T,
        torch.full_like(y_true, alpha),
        torch.ones_like(y_true)
    )

    # apply weights
    return (weights * base_loss).mean()

In [ ]:
# Here we have the same process, but instead of 2 steps we have 4 to provide a more nuanced weighting scheme.
# In this case, T1, T2, and T3 are the benchmark values, and alpha1, alpha2, alpha3, and alpha4 are the weights for each range. The T
# values are embedded into the function as they are less likely to change than the alpha values.
def weighted_loss(y_pred, y_true, alpha1, alpha2, alpha3, alpha4):
    # Calculate log benchmarks
    T1 = torch.log(torch.tensor(5.0))    # ln(5)
    T2 = torch.log(torch.tensor(50.0))   # ln(50)
    T3 = torch.log(torch.tensor(100.0))  # ln(100)

    # Base loss (MSE here, but could be MAE or Huber)
    base_loss = (y_pred - y_true) ** 2

    # Weight function based on y_true ranges
    weights = torch.where(
        y_true <= T1,
        torch.full_like(y_true, alpha1),  # First range (y_true <= ln(5))
        torch.where(
            y_true <= T2,
            torch.full_like(y_true, alpha2),  # Second range (ln(5) < y_true <= ln(50))
            torch.where(
                y_true <= T3,
                torch.full_like(y_true, alpha3),  # Third range (ln(50) < y_true <= ln(100))
                torch.full_like(y_true, alpha4)   # Fourth range (y_true > ln(100))
            )
        )
    )

    # Apply weights to the base loss
    return (weights * base_loss).mean()

Here is the application of the loss function to our model architecture:

In [ ]:
# The condinonal encoder architecture is going to remain largely unchanged in this process since the chanages are really only applied
# to the loss function itself:
class Cond_Encoder_1234(nn.Module):
    def __init__(self, input_size, output_size, num_layers):
        super().__init__()
        layers = []
        layer_sizes = np.linspace(input_size, output_size, num_layers + 1, dtype=int)
        for i in range(num_layers):
            layers.append(nn.Linear(layer_sizes[i], layer_sizes[i+1]))
            if i < num_layers - 1:
                layers.append(nn.LeakyReLU(inplace=True))
        self.encoder = nn.Sequential(*layers)

    def forward(self, x):
        output = self.encoder(x)
        
        # Split the output into four parts
        embedding_output = output[:, :512]    # ChemNet embeddings (no activation)
        toxicity_raw = output[:, 512:513]     # Raw toxicity output (1 column)
        morgan_output = output[:, 513:513+2048]  # Morgan fingerprints (2048 columns)
        filtered_morgan_output = output[:, 513+2048:]  # Filtered Morgan fingerprints (remaining columns)
        
        # Apply scaled sigmoid only to toxicity part
        toxicity_output = torch.sigmoid(toxicity_raw) * np.log(100000) 

        # Concatenate back together
        final_output = torch.cat([embedding_output, toxicity_output, morgan_output, filtered_morgan_output], dim=1)
        
        return final_output

In [ ]:
# The full model traiing function with the 2 step weighted loss function incorportated

def train_model_condenc_1234e1e2_weightloss(model, train_data, val_data, epochs, learning_rate, criterion1, criterion3, criterion4,
                                 lambda1, lambda2, lambda3, lambda4, device, config=None, 
                                 weighted_loss_params=None):
    # Initialize optimizer
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    # Initialize wandb and log parameters
    wandb.init(entity=config['wandb_entity'],
               project=config['wandb_project'],
               config=config)
    wandb.config.update(weighted_loss_params)  # Log the weighted loss parameter "alpha"

    # Extract `alpha` for weighted loss
    alpha = weighted_loss_params['alpha']

    # Lists for storing training and validation losses for monitoring
    train_losses, val_losses = [], []
    train_embedding_losses, train_toxicity_losses, train_morgan_losses, train_filtered_morgan_losses = [], [], [], []
    val_embedding_losses, val_toxicity_losses, val_morgan_losses, val_filtered_morgan_losses = [], [], [], []

    for epoch in range(epochs):
        model.train()  # Set the model to training mode

        # Accumulate training losses
        running_loss = 0.0
        running_embedding_loss = 0.0
        running_toxicity_loss = 0.0
        running_morgan_loss = 0.0
        running_filtered_morgan_loss = 0.0

        for batch_with_ext, true_embeddings, true_log_tox, true_morgan, true_filtered_morgan, _ in train_data:
            # Move data to the specified device
            batch_with_ext = batch_with_ext.to(device)
            true_embeddings = true_embeddings.to(device)
            true_log_tox = true_log_tox.to(device)
            true_morgan = true_morgan.to(device)
            true_filtered_morgan = true_filtered_morgan.to(device)

            # Zero gradients
            optimizer.zero_grad()

            # Forward pass
            batch_predicted_combined = model(batch_with_ext)

            # Embedding Loss
            batch_predicted_embeddings = batch_predicted_combined[:, :512]
            loss1 = criterion1(batch_predicted_embeddings, true_embeddings)

            # Toxicity Loss (using the updated weighted_loss function)
            batch_predicted_log_tox = batch_predicted_combined[:, 512:513]
            loss2 = weighted_loss(batch_predicted_log_tox, true_log_tox, alpha)

            # Morgan Loss
            batch_predicted_morgan = batch_predicted_combined[:, 513:513 + 2048]
            loss3 = criterion3(batch_predicted_morgan, true_morgan)

            # Filtered Morgan Loss
            batch_predicted_filtered_morgan = batch_predicted_combined[:, 513 + 2048:]
            loss4 = criterion4(batch_predicted_filtered_morgan, true_filtered_morgan)

            # Apply Lambda Scaling
            weighted_loss1 = lambda1 * loss1
            weighted_loss2 = lambda2 * loss2
            weighted_loss3 = lambda3 * loss3
            weighted_loss4 = lambda4 * loss4

            # Total Loss
            total_loss = weighted_loss1 + weighted_loss2 + weighted_loss3 + weighted_loss4

            # Backpropagation and optimizer step
            total_loss.backward()
            optimizer.step()

            # Accumulate losses for this batch
            running_loss += total_loss.item()
            running_embedding_loss += weighted_loss1.item()
            running_toxicity_loss += weighted_loss2.item()
            running_morgan_loss += weighted_loss3.item()
            running_filtered_morgan_loss += weighted_loss4.item()

        # Store and log epoch-wise averaged training losses
        train_losses.append(running_loss / len(train_data))
        train_embedding_losses.append(running_embedding_loss / len(train_data))
        train_toxicity_losses.append(running_toxicity_loss / len(train_data))
        train_morgan_losses.append(running_morgan_loss / len(train_data))
        train_filtered_morgan_losses.append(running_filtered_morgan_loss / len(train_data))

        wandb.log({
            "epoch": epoch,
            "train_loss": train_losses[-1],
            "train_embedding_loss": train_embedding_losses[-1],
            "train_toxicity_loss": train_toxicity_losses[-1],
            "train_morgan_loss": train_morgan_losses[-1],
            "train_filtered_morgan_loss": train_filtered_morgan_losses[-1]
        })

        # Validation Phase
        model.eval()  # Set the model to evaluation mode
        val_loss = 0.0
        val_running_embedding_loss = 0.0
        val_running_toxicity_loss = 0.0
        val_running_morgan_loss = 0.0
        val_running_filtered_morgan_loss = 0.0

        with torch.no_grad():
            for val_batch_with_ext, val_true_embeddings, val_true_tox, val_true_morgan, val_true_filtered_morgan, _ in val_data:
                # Move validation data to the specified device
                val_batch_with_ext = val_batch_with_ext.to(device)
                val_true_embeddings = val_true_embeddings.to(device)
                val_true_tox = val_true_tox.to(device)
                val_true_morgan = val_true_morgan.to(device)
                val_true_filtered_morgan = val_true_filtered_morgan.to(device)

                # Forward pass
                val_batch_predicted = model(val_batch_with_ext)

                # Embedding Loss
                val_predicted_embeddings = val_batch_predicted[:, :512]
                val_loss1 = criterion1(val_predicted_embeddings, val_true_embeddings)

                # Toxicity Loss (using the updated weighted_loss function)
                val_predicted_tox = val_batch_predicted[:, 512:513]
                val_loss2 = weighted_loss(val_predicted_tox, val_true_tox, alpha)

                # Morgan Loss
                val_predicted_morgan = val_batch_predicted[:, 513:513 + 2048]
                val_loss3 = criterion3(val_predicted_morgan, val_true_morgan)

                # Filtered Morgan Loss
                val_predicted_filtered_morgan = val_batch_predicted[:, 513 + 2048:]
                val_loss4 = criterion4(val_predicted_filtered_morgan, val_true_filtered_morgan)

                # Apply Lambda Scaling
                val_weighted_loss1 = lambda1 * val_loss1
                val_weighted_loss2 = lambda2 * val_loss2
                val_weighted_loss3 = lambda3 * val_loss3
                val_weighted_loss4 = lambda4 * val_loss4

                # Total Validation Loss
                val_loss += (val_weighted_loss1 + val_weighted_loss2 + val_weighted_loss3 + val_weighted_loss4).item()
                val_running_embedding_loss += val_weighted_loss1.item()
                val_running_toxicity_loss += val_weighted_loss2.item()
                val_running_morgan_loss += val_weighted_loss3.item()
                val_running_filtered_morgan_loss += val_weighted_loss4.item()

        # Store and log validation losses
        val_losses.append(val_loss / len(val_data))
        val_embedding_losses.append(val_running_embedding_loss / len(val_data))
        val_toxicity_losses.append(val_running_toxicity_loss / len(val_data))
        val_morgan_losses.append(val_running_morgan_loss / len(val_data))
        val_filtered_morgan_losses.append(val_running_filtered_morgan_loss / len(val_data))

        wandb.log({
            "epoch": epoch,
            "val_loss": val_losses[-1],
            "val_embedding_loss": val_embedding_losses[-1],
            "val_toxicity_loss": val_toxicity_losses[-1],
            "val_morgan_loss": val_morgan_losses[-1],
            "val_filtered_morgan_loss": val_filtered_morgan_losses[-1]
        })

    # Finish wandb session
    wandb.finish()

    return model, {
        "train_losses": train_losses,
        "val_losses": val_losses,
        "train_embedding_losses": train_embedding_losses,
        "val_embedding_losses": val_embedding_losses,
        "train_toxicity_losses": train_toxicity_losses,
        "val_toxicity_losses": val_toxicity_losses
    }

Here is the full trianing function code with the weighted loss impleneted inside:

In [ ]:
def train_model_condenc_1234e1e2(model, train_data, val_data, epochs, learning_rate, criterion1, criterion3, criterion4,
                                 lambda1, lambda2, lambda3, lambda4, device, config=None,
                                 alpha1=2, alpha2=1.5, alpha3=1.0, alpha4=0.5):
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    wandb.init(entity=config['wandb_entity'],
               project=config['wandb_project'],
               config=config)

    # Initialize lists for storing losses
    train_losses = []
    val_losses = []
    train_embedding_losses, train_toxicity_losses, train_morgan_losses, train_filtered_morgan_losses = ([] for _ in range(4))
    val_embedding_losses, val_toxicity_losses, val_morgan_losses, val_filtered_morgan_losses = ([] for _ in range(4))

    # Log weighted loss parameters
    wandb.config.update({
        "alpha1": alpha1, "alpha2": alpha2, "alpha3": alpha3, "alpha4": alpha4
    })

    for epoch in range(epochs):
        # Training Mode
        model.train()
        running_loss = 0.0
        running_embedding_loss, running_toxicity_loss = 0.0, 0.0
        running_morgan_loss, running_filtered_morgan_loss = 0.0, 0.0

        for batch_with_ext, true_embeddings, true_log_tox, true_morgan, true_filtered_morgan, _ in train_data:
            batch_with_ext = batch_with_ext.to(device)  # Input includes spectra + group encoding + collision energy encoding
            true_embeddings = true_embeddings.to(device)
            true_log_tox = true_log_tox.to(device)
            true_morgan = true_morgan.to(device)
            true_filtered_morgan = true_filtered_morgan.to(device)

            optimizer.zero_grad()
            batch_predicted_combined = model(batch_with_ext)  # Forward pass with external conditions info

            # Embedding Loss
            batch_predicted_embeddings = batch_predicted_combined[:, :512]  # First 512 columns
            loss1 = criterion1(batch_predicted_embeddings, true_embeddings)  # loss1 (embedding loss)

            # Toxicity Loss (4-step weighted_loss)
            batch_predicted_log_tox = batch_predicted_combined[:, 512:513]  # 512th column
            loss2 = weighted_loss(batch_predicted_log_tox, true_log_tox, alpha1, alpha2, alpha3, alpha4)  # Use 4-step weighted loss

            # Morgan Loss
            batch_predicted_morgan = batch_predicted_combined[:, 513:513 + 2048]  # Next 2048 columns
            loss3 = criterion3(batch_predicted_morgan, true_morgan)  # loss3 (morgan loss)

            # Filtered Morgan Loss
            batch_predicted_filtered_morgan = batch_predicted_combined[:, 513 + 2048:]  # Remaining columns
            loss4 = criterion4(batch_predicted_filtered_morgan, true_filtered_morgan)  # loss4 (filtered morgan loss)

            # Apply lambda weighting
            weighted_loss1 = lambda1 * loss1
            weighted_loss2 = lambda2 * loss2
            weighted_loss3 = lambda3 * loss3
            weighted_loss4 = lambda4 * loss4

            # Total loss with modular weights
            total_loss = weighted_loss1 + weighted_loss2 + weighted_loss3 + weighted_loss4
            total_loss.backward()
            optimizer.step()

            # Accumulate losses
            running_loss += total_loss.item()
            running_embedding_loss += weighted_loss1.item()
            running_toxicity_loss += weighted_loss2.item()
            running_morgan_loss += weighted_loss3.item()
            running_filtered_morgan_loss += weighted_loss4.item()

        # Calculate average train losses
        average_train_loss = running_loss / len(train_data)
        average_train_embedding_loss = running_embedding_loss / len(train_data)
        average_train_toxicity_loss = running_toxicity_loss / len(train_data)
        average_train_morgan_loss = running_morgan_loss / len(train_data)
        average_train_filtered_morgan_loss = running_filtered_morgan_loss / len(train_data)

        wandb.log({
            "average_train_loss": average_train_loss,
            "average_train_embedding_loss": average_train_embedding_loss,
            "average_train_toxicity_loss": average_train_toxicity_loss,
            "average_train_morgan_loss": average_train_morgan_loss,
            "average_train_filtered_morgan_loss": average_train_filtered_morgan_loss
        })

        # Validation Mode
        model.eval()
        val_loss = 0.0
        val_embedding_loss, val_toxicity_loss = 0.0, 0.0
        val_morgan_loss, val_filtered_morgan_loss = 0.0, 0.0

        with torch.no_grad():
            for val_batch_with_ext, val_true_embeddings, val_true_tox, val_true_morgan, val_true_filtered_morgan, _ in val_data:
                val_batch_with_ext = val_batch_with_ext.to(device)
                val_true_embeddings = val_true_embeddings.to(device)
                val_true_tox = val_true_tox.to(device)
                val_true_morgan = val_true_morgan.to(device)
                val_true_filtered_morgan = val_true_filtered_morgan.to(device)

                val_batch_predicted = model(val_batch_with_ext)
                val_batch_predicted_embeddings = val_batch_predicted[:, :512]
                val_batch_predicted_tox = val_batch_predicted[:, 512:513]
                val_batch_predicted_morgan = val_batch_predicted[:, 513:513 + 2048]
                val_batch_predicted_filtered_morgan = val_batch_predicted[:, 513 + 2048:]

                # Calculate individual validation losses
                val_loss1 = criterion1(val_batch_predicted_embeddings, val_true_embeddings)
                val_loss2 = weighted_loss(val_batch_predicted_tox, val_true_tox, alpha1, alpha2, alpha3, alpha4)  # Weighted loss
                val_loss3 = criterion3(val_batch_predicted_morgan, val_true_morgan)
                val_loss4 = criterion4(val_batch_predicted_filtered_morgan, val_true_filtered_morgan)

                # Apply lambda weighting
                val_weighted_loss1 = lambda1 * val_loss1
                val_weighted_loss2 = lambda2 * val_loss2
                val_weighted_loss3 = lambda3 * val_loss3
                val_weighted_loss4 = lambda4 * val_loss4

                # Total weighted val loss
                val_loss += (val_weighted_loss1 + val_weighted_loss2 + val_weighted_loss3 + val_weighted_loss4).item()
                val_embedding_loss += val_weighted_loss1.item()
                val_toxicity_loss += val_weighted_loss2.item()
                val_morgan_loss += val_weighted_loss3.item()
                val_filtered_morgan_loss += val_weighted_loss4.item()

        # Calculate average validation losses
        average_val_loss = val_loss / len(val_data)
        wandb.log({
            "average_val_loss": average_val_loss
        })

        if epoch % 10 == 0 or epoch == epochs - 1:
            print(f"Epoch [{epoch+1}/{epochs}]")
            print(f"Training loss: {average_train_loss:.6f}")
            print(f"Validation loss: {average_val_loss:.6f}")

    wandb.finish()
    return model, train_losses, val_losses, train_embedding_losses, train_toxicity_losses, train_morgan_losses, train_filtered_morgan_losses, val_embedding_losses, val_toxicity_losses, val_morgan_losses, val_filtered_morgan_losses